# Notebook Instructions

1. If you are new to Jupyter Notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.

# Backtest and Trade-Level Analytics of GARCH Forecast

In the previous notebook, you learnt to generate trading signals based on volatility forecasted by Garch(1,1) model. In the next notebook, you will learn to backtest and perform trade-level analytics on the signals generated in the previous notebook. 

1. [Read the Data](#data)
2. [Backtest and Performance Analysis](#backtest)
3. [Trade Level Analytics](#trade_level_analytics)
4. [Conclusion](#conclusion)


## Import Libraries

In [1]:
# For data manipulation
import numpy as np
import pandas as pd

# Helper functions
import sys
sys.path.append("..")
from data_modules.options_volatility_utils import trade_level_analytics

<a id='data'></a>
## Read the Data

Import the file `spy_volatility_forecast_trading_signals.csv` using the `read_csv` method of `pandas`. It has the details of all monthly expiry options contracts available for S&P 500 index from 2018 till 2022 along with the trading signals based on the forecasted volatility by the GARCH(1,1) model. <br><br>The data includes the strike price of the options contract, the implied volatility and the price of calls, puts of the strike price, the open, high, low, and close prices of underlying i.e. S&P 500 index for the day and trading signals.  <br><br>This CSV file is available in the zip file of the unit 'Python Codes and Data' in the 'Course Summary' section.

In [2]:
# Read the data
options_data = pd.read_csv(
    '../data_modules/spy_volatility_forecast_trading_signals.csv', index_col=0)
options_data.tail()

,STRIKE,C_IV,P_IV,C_LAST,P_LAST,Open,High,Low,Close,signal
[QUOTE_DATE],,,,,,,,,,
2022-09-23,3695.0,0.28858,0.28858,57.8,66.68,3727.139893,3727.139893,3647.469971,3693.229980,-1
2022-09-26,3655.0,0.34657,0.34657,64.9,52.00,3682.719971,3715.669922,3644.760010,3655.040039,-1
2022-09-27,3650.0,0.33897,0.33897,48.0,47.66,3686.439941,3717.530029,3623.290039,3647.290039,-1
2022-09-28,3720.0,0.27130,0.27130,40.1,31.90,3651.939941,3736.739990,3640.610107,3719.040039,0
2022-09-29,3640.0,0.30141,0.30141,33.8,28.20,3687.010010,3687.010010,3610.399902,3640.469971,0


<a id='backtest'></a>
## Backtest and Performance Analysis

We will loop over each of the dates in the trading signals data, set up the straddle when the signal is 1 or -1, exit when the signal is 0, and update the trade details in `round_trips_details`. 

The `mark_to_market` dataframe contains the premiums of the strategy on each date between the entry date and exit date.

We will backtest the trading signals of straddle using the following steps:

**Step-1**: Create dataframes `round_trips_details`, `trades` and `mark_to_market` for storing round trips, trades, and mtm, respectively.

**Step-2**: Define a function `add_to_mtm` which stores daily mark_to_market values for the strategy. It takes the existing `mark_to_market` dataframe, `option_strategy` which is a straddle in this case and `trading_date` as inputs.

**Step-3**: Define a function `get_premium` to get the premium for the two legs(CE and PE) of the straddle. It takes `options_straddle` and `options_data` as inputs.

**Step-4**: Define a function `setup_straddle` to set up the straddle strategy. It takes `options_data` and `direction` (long/short) as inputs.

**Step-5**: Initialise `current_position`, `trade_num`, which is basically the number of trades, `cum_pnl` to 0 and set the `entry_flag` to `False`. 

**Step-6**: We also set the `start_date` for backtesting.

In [3]:
# Step-1: Create dataframes for round trips, storing trades, and mtm
round_trips_details = pd.DataFrame()
trades = pd.DataFrame()
mark_to_market = pd.DataFrame()

# Step-2: Function for calculating mtm
def add_to_mtm(mark_to_market, option_strategy, trading_date):    
    
    # Create a 'Date' column to store the trading date
    option_strategy['Date'] = trading_date
    
    # Create the mark_to_market dataframe
    mark_to_market = pd.concat([mark_to_market, option_strategy])
    
    # Return the mark_to_market dataframe
    return mark_to_market

# Step-3: Function for fetching premium
def get_premium(options_strategy, options_data):    
    
    # Get the premium for call option
    if options_strategy['Option Type'] == "CE":
        
        # Return the premium of call option
        return options_data['C_LAST']

    # Get the premium for put option
    elif options_strategy['Option Type'] == "PE":
        
        # Return the premium of put option
        return options_data['P_LAST']

# Step-4: Function for setting up a straddle
def setup_straddle(options_data, direction='short'):

    # Create a dataframe to store the straddle
    straddle = pd.DataFrame()

    # CE and PE legs of the straddle
    straddle['Option Type'] = ['CE', 'PE']

    # Create the straddle at ATM
    straddle['Strike Price'] = options_data.STRIKE

    # Sell positions for both CE and PE legs in case of a short straddle
    straddle['position'] = -1

    # Get the premiums for the two option legs of the short straddle
    straddle['premium'] = straddle.apply(
        lambda r: get_premium(r, options_data), axis=1)

    # Multiply the position by -1 in case of a long straddle
    if direction == 'long':
        straddle['position'] *= -1

        # Get the premiums for the two option legs of the long straddle
        straddle['premium'] = straddle.apply(
            lambda r: get_premium(r, options_data), axis=1)
    
    # Return the straddle dataframe
    return straddle

# Step-5: Initialise current position, number of trades and cumulative pnl to 0
current_position = 0
trade_num = 0
cum_pnl = 0

# Set exit flag to False
exit_flag = False

# Step-6: Set start date for backtesting
start_date = options_data.index[0]

Perform the following steps iteratively for the dates in the backtest period.

**Step-7**: For a given date, if there is no open position and the signal is '1' or '-1', we will set up the straddle.

**Step-8**: For a given date, if there is an open position, we exit the trade if the signal is '0'.

**Step-9**: Finally, we calculate the pnl for each trade and also the cumulative pnl.

In [4]:
# Run the loop for every date in the `options_data` dataframe
for i in options_data.loc[start_date:].index:

# Step-7: Check the entry conditions
    # Check condition to open long strangle when the current position is '0' and 'signal' is '1'
    if (current_position == 0) and options_data.loc[i, 'signal'] == 1:

        # Setup long straddle
        straddle = setup_straddle(options_data.loc[i], direction="long")

        # Populate the 'trades' dataframe
        trades = straddle.copy()
        trades['entry_date'] = i
        trades.rename(columns={'premium': 'entry_price'}, inplace=True)

        # Calculate net premium
        net_premium = round((straddle.position * straddle.premium).sum(), 1)

        # Update current position to 1 since a long straddle position is opened
        current_position = 1

        # Update mark_to_market dataframe
        mark_to_market = add_to_mtm(mark_to_market, straddle, i)

        # Increase number of trades by 1
        trade_num += 1
        print("-"*100)

        # Print trade details
        print(
            f"Trade No: {trade_num} | Long Strangle | Entry | Date: {i} | Premium: {net_premium} | Pnl: 0 | Cum PnL: {cum_pnl}")

    # Check condition to open short strangle when current the position is '0' and 'signal' is '-1'
    elif (current_position == 0) and options_data.loc[i, 'signal'] == -1:

        # Setup straddle
        straddle = setup_straddle(options_data.loc[i], direction="short")

        # Populate the trades dataframe
        trades = straddle.copy()
        trades['entry_date'] = i
        trades.rename(columns={'premium': 'entry_price'}, inplace=True)

        # Calculate net premium
        net_premium = round((straddle.position * straddle.premium).sum(), 1)

        # Update current position to -1
        current_position = -1

        # Update mark_to_market dataframe
        mark_to_market = add_to_mtm(mark_to_market, straddle, i)

        # Increase number of trades by 1
        trade_num += 1
        print("-"*100)

        # Print trade details
        print(
            f"Trade No: {trade_num} | Short Straddle  | Entry | Date: {i} | Premium: {net_premium} | Pnl: 0 | Cum PnL: {cum_pnl}")

# Step-8: Check the exit condition    
    # Check condition to close the trades when the current position is '1' or '-1'
    elif current_position == 1 or current_position == -1:

        # Update net premium
        straddle['premium'] = straddle.apply(
            lambda r: get_premium(r, options_data.loc[i]), axis=1)
        net_premium = (straddle.position * straddle.premium).sum()

        # Update mark_to_market dataframe
        mark_to_market = add_to_mtm(mark_to_market, straddle, i)

        # Exit the trade if any of the exit condition is met
        if options_data.loc[i, 'signal'] == 0:

            # Append the trades dataframe
            trades['exit_date'] = i
            trades['exit_price'] = straddle.premium

            # Calculate net premium at exit
            net_premium = round(
                (straddle.position * straddle.premium).sum(), 1)

            # Calculate net premium on entry
            entry_net_premium = (trades.position * trades.entry_price).sum()

# Step-9: Calculate the pnl for each trade and also the cumulative pnl
            # Calculate pnl for the trade
            trade_pnl = round(net_premium - entry_net_premium, 1)

            # Add the trade logs to round trip details
            round_trips_details = pd.concat([round_trips_details, trades])
          
            # Calculate cumulative pnl
            cum_pnl += trade_pnl
            cum_pnl = round(cum_pnl, 2)

            # Print trade details
            print(
                f"Trade No: {trade_num} | Date: {i} | Premium: {net_premium} | PnL: {trade_pnl} | Cum PnL: {cum_pnl}")

            # Update current position to 0
            current_position = 0

            # Set exit flag to false
            exit_flag = False

----------------------------------------------------------------------------------------------------
Trade No: 1 | Short Straddle  | Entry | Date: 2022-01-24 | Premium: -242.9 | Pnl: 0 | Cum PnL: 0
Trade No: 1 | Date: 2022-01-31 | Premium: -2.7 | PnL: 240.2 | Cum PnL: 240.2
----------------------------------------------------------------------------------------------------
Trade No: 2 | Short Straddle  | Entry | Date: 2022-02-01 | Premium: -266.8 | Pnl: 0 | Cum PnL: 240.2
Trade No: 2 | Date: 2022-02-08 | Premium: -152.1 | PnL: 114.7 | Cum PnL: 354.9
----------------------------------------------------------------------------------------------------
Trade No: 3 | Short Straddle  | Entry | Date: 2022-02-09 | Premium: -124.9 | Pnl: 0 | Cum PnL: 354.9
Trade No: 3 | Date: 2022-02-16 | Premium: -125.7 | PnL: -0.8 | Cum PnL: 354.1
----------------------------------------------------------------------------------------------------
Trade No: 4 | Short Straddle  | Entry | Date: 2022-02-17 | Prem

In [5]:
# Round trip details
round_trips_details.head()

,Option Type,Strike Price,position,entry_price,entry_date,exit_date,exit_price
0,CE,4410.0,-1,73.24,2022-01-24,2022-01-31,1.55
1,PE,4410.0,-1,169.66,2022-01-24,2022-01-31,1.15
0,CE,4545.0,-1,82.72,2022-02-01,2022-02-08,72.28
1,PE,4545.0,-1,184.08,2022-02-01,2022-02-08,79.83
0,CE,4585.0,-1,56.79,2022-02-09,2022-02-16,61.80


<a id='trade_level_analytics'></a>
## Trade Level Analytics

Use the `trade_level_analysis` function to generate the trade level analytics of the trade. It takes the `round_trips` dataframe and `lot_size` as the input. <br>

The following are the output of the `trade_level_analysis` function. 
* Total PnL	
* Total Trades
* Number of Winners
* Number of Losers
* Win (%)
* Loss (%)
* Per Trade PnL of Winners
* Per Trade PnL of Losers
* Profit Factor

You can find the function in the `options_volatility_utils.py` file in the `data_modules` folder of the zip file.

In [6]:
# Trade Level Analytics
# Assume lot size as 5
analytics = trade_level_analytics(round_trips_details, lot_size= 5)
analytics

,Strategy
Total PnL,5209.800000
Total Trades,27.000000
Number of Winners,21.000000
Number of Losers,6.000000
Win (%),77.777778
Loss (%),22.222222
Per Trade PnL of Winners,318.204762
Per Trade PnL of Losers,245.416667
Profit Factor,4.538065


The trading strategy based on volatility forecasted by the GARCH(1,1) model has the win rate of `77.77%` and the profit factor of `4.5`. This indicates that the strategy has been significantly profitable, with profits outweighing losses by a substantial margin.  
<a id='conclusion'></a>
## Conclusion

In this notebook, we backtested the trading signals generated based on volatility forecasted by GARCH(1,1) model. The trades were exited after holding for a week. You can extend the exit criteria by adding stop-loss and take-profit to improve the performance of the strategy. <br><br>